<a href="https://colab.research.google.com/github/bsheese/387/blob/main/19_onramp/19_1_ML_Foundations/19_1_5_What_One_Neuron_Cant_Do.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# The On-Ramp: Part 5
## What One Neuron Can't Do

---

## What This Notebook Is About

19_1_3 showed that a neuron computes a weighted sum of its inputs plus a bias, and that logistic regression passes that sum through sigmoid to get a probability. 19_1_4 showed that the Pets model's last layer is four of these weighted sums, turned into probabilities by softmax. This notebook asks whether a weighted sum is always enough to separate the answers, using a problem small enough to draw.

1. **XOR**: a tiny problem with four points that no single neuron can solve
2. **A hidden layer**: two neurons feeding a third solve XOR, which is the reason networks have more than one layer

Nothing here needs the Pets files or a GPU, and every cell runs in a second or two.

**Optional reference.** If you took CS 377, you have seen these ideas before in [18_2_3](https://github.com/bsheese/cs377/blob/main/18_classification/18_2_LogisticRegression/18_2_3_LogReg_to_NeuralNetworks.ipynb). If you haven't, that notebook covers the same ideas in more depth. You don't need it for this course.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LogisticRegression

---

## Section 1: A Problem One Neuron Can't Solve

Here is a tiny made-up problem with two features and four points. The answer is 1 when exactly one of the two features is 1, and 0 otherwise. This pattern is called **XOR** ("exclusive or"). Unit 20 uses these same four points.

In [ ]:
X_xor = np.array([[0, 0],
                  [0, 1],
                  [1, 0],
                  [1, 1]])
y_xor = np.array([0, 1, 1, 0])

for x, y in zip(X_xor, y_xor):
    print(f'x1 = {x[0]}, x2 = {x[1]}  ->  {y}')

In [ ]:
def plot_xor(ax, title):
    for x, y in zip(X_xor, y_xor):
        ax.scatter(x[0], x[1], s=250, c='#C0392B' if y == 1 else '#2980B9', edgecolors='black', zorder=3)
        ax.annotate(str(y), (x[0], x[1]), xytext=(10, 10), textcoords='offset points', fontsize=13)
    ax.set_xlim(-0.5, 1.5)
    ax.set_ylim(-0.5, 1.5)
    ax.set_xlabel('x1')
    ax.set_ylabel('x2')
    ax.set_title(title)

fig, ax = plt.subplots(figsize=(4.5, 4.5))
plot_xor(ax, 'XOR: red = 1, blue = 0')
plt.show()

The two 1s sit on opposite corners of the square, and so do the two 0s.

A single neuron computes $z = w_1 x_1 + w_2 x_2 + b$ and answers 1 when the probability is above 0.5, which is when $z > 0$ (sigmoid of 0 is exactly 0.5). The points where $z = 0$ form a **straight line**. So whatever weights a single neuron has, it splits the plane with one straight line and answers 1 on one side and 0 on the other. The line is called its **decision boundary**.

The next cell draws the boundary for three choices of weights and counts how many of the four points each one gets right.

In [ ]:
def neuron_answers(w1, w2, b):
    z = w1 * X_xor[:, 0] + w2 * X_xor[:, 1] + b
    return (z > 0).astype(int)

attempts = [(1, 1, -0.5), (-1, -1, 1.5), (1, -1, -0.5)]
xs = np.linspace(-0.5, 1.5, 10)
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
for ax, (w1, w2, b) in zip(axes, attempts):
    right = (neuron_answers(w1, w2, b) == y_xor).sum()
    plot_xor(ax, f'w1={w1}, w2={w2}, b={b}:  {right} of 4 right')
    ax.plot(xs, -(w1 * xs + b) / w2, 'k--')    # the line where z = 0
plt.tight_layout()
plt.show()

Each line gets 3 of the 4 points right. To get all four, a line would have to put the two red corners on one side and the two blue corners on the other, and no straight line can do that, because the red corners are diagonally opposite each other. Trying thousands of weights confirms it:

In [ ]:
best = 0
grid = np.linspace(-3, 3, 25)
for w1 in grid:
    for w2 in grid:
        for b in grid:
            best = max(best, (neuron_answers(w1, w2, b) == y_xor).sum())
print(f'tried {len(grid) ** 3:,} combinations of weights; the best got {best} of 4 right')

Logistic regression, the trained single neuron from 19_1_3, does even worse:

In [ ]:
lr = LogisticRegression().fit(X_xor, y_xor)
print('weights:', lr.coef_[0].round(3), '  bias:', lr.intercept_.round(3))
print('probabilities of 1:', lr.predict_proba(X_xor)[:, 1].round(3))
print('accuracy:', lr.score(X_xor, y_xor))

It sets both weights to 0 and gives every point a probability of 0.5. Every line gets at least one point wrong, and because the problem is symmetric there's no reason to prefer one line over another, so the best it can do by the cross-entropy loss from 19_1_4 is to say "50/50" everywhere. Its accuracy of 0.5 is no better than guessing.

This is not just a puzzle. In 1969 Marvin Minsky and Seymour Papert pointed out that a single-layer network can't learn XOR, and their book helped turn many researchers away from neural networks for over a decade.

---

## Section 2: Adding a Hidden Layer

The way around the problem is to put neurons in front of the neuron that gives the answer. We can build such a network by hand. To keep the arithmetic easy, each neuron here uses a **step** activation instead of sigmoid: it outputs 1 if its weighted sum is above 0 and 0 otherwise.

- **Hidden neuron 1** has weights (1, 1) and bias −0.5. It outputs 1 when **at least one** feature is 1.
- **Hidden neuron 2** has weights (1, 1) and bias −1.5. It outputs 1 only when **both** features are 1.
- **The output neuron** takes the two hidden outputs as its inputs, with weights (1, −1) and bias −0.5. It outputs 1 when hidden neuron 1 is on and hidden neuron 2 is off, which is "at least one, but not both".

In [ ]:
def step(z):
    return (z > 0).astype(int)

h1 = step(X_xor @ np.array([1, 1]) - 0.5)    # at least one is 1
h2 = step(X_xor @ np.array([1, 1]) - 1.5)    # both are 1
H = np.column_stack([h1, h2])                # the hidden layer's outputs, one row per point
out = step(H @ np.array([1, -1]) - 0.5)      # at least one, but not both

print('x1 x2 | h1 h2 | output  target')
for x, h, o, y in zip(X_xor, H, out, y_xor):
    print(f' {x[0]}  {x[1]} |  {h[0]}  {h[1]} |   {o}       {y}')

All four points are right. The two **hidden neurons** form a **hidden layer**. It's called hidden because its outputs are neither the inputs nor the final answer; they are new features that the network computes for itself.

Plotting the points by their hidden-layer values, instead of their original features, shows why this works:

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4.5))
plot_xor(ax1, 'Original features: no line works')

for h, y in zip(H, y_xor):
    ax2.scatter(h[0], h[1], s=250, c='#C0392B' if y == 1 else '#2980B9', edgecolors='black', zorder=3)
ax2.annotate('(0, 1) and (1, 0)\nboth land here', (1, 0), xytext=(0.3, 0.35), fontsize=10,
             arrowprops=dict(arrowstyle='->'))
xs = np.linspace(-0.5, 1.5, 10)
ax2.plot(xs, xs - 0.5, 'k--')     # the output neuron's line: h1 - h2 - 0.5 = 0
ax2.set_xlim(-0.5, 1.5)
ax2.set_ylim(-0.5, 1.5)
ax2.set_xlabel('h1 (at least one)')
ax2.set_ylabel('h2 (both)')
ax2.set_title("Hidden-layer features: one line works")
plt.tight_layout()
plt.show()

In the hidden layer's features, the two red points land on the same spot, and a single straight line separates them from the two blue points. The hidden layer doesn't solve the problem by itself. It moves the points into a new arrangement where the output neuron's straight line is enough.

That is the main idea behind deep networks, and it applies directly to our Pets model. Its last layer is four neurons, and each one can only draw straight boundaries through the 512 features. That works because the 17 layers in front of it turn photos, where breeds are hopelessly tangled together, into 512 features where breeds can be separated with straight boundaries. Those earlier layers are hidden layers too, a lot of them.

We set the hidden weights here by hand, because we already knew the answer. In unit 20 a network learns the weights of its hidden layer from the four XOR points on its own, and in 22_1 we look at what ResNet's hidden layers have learned.

---

## Summary

- A single neuron's **decision boundary** is a straight line (with more features, a flat surface). **XOR** can't be separated by any straight line, so no single neuron solves it; logistic regression gives up and answers 0.5 everywhere.
- A **hidden layer** of neurons computes new features. With two hidden neurons ("at least one" and "both"), the XOR points can be separated by a straight line, and the network gets all four right.
- The Pets model works the same way: its hidden layers turn photos into 512 features in which the breeds can be separated by the last layer's straight boundaries.

## What's next

The hidden weights in this notebook were set by hand, because we already knew the answer. A real network has to find its weights on its own, starting from random numbers, and unit 20 shows how. 20_0_1 introduces tensors, PyTorch's version of NumPy arrays, and then asks how to tell which way to change a weight: change it by a tiny amount and measure how the loss responds, and then let PyTorch work out the same thing for every weight at once. By 20_1_1, a network with a hidden layer learns the XOR weights by itself.